# 使用するライブラリの読み込み

In [1]:
import datetime
import glob
from io import StringIO
import os
import random
import re
import time

import requests
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions
from selenium.webdriver.chrome.options import Options

import chromedriver_binary

# レース結果のRAWデータ作成
`makeRawData.py`の`makeRaceData`クラスとしてモジュール化済み  
レース結果のHTMLを取得し、[レースID].txtとしてdata/html/raceに保存。  
保存したHTMLから  
- レース結果を[レースID].pickleとしてdata/rawData/raceResultsに
- レース情報を[レースID].pickleとしてdata/rawData/raceInfosに
- 払い戻し情報を[レースID].pickleとしてdata/rawData/returnsに  
それぞれ保存する



## レース結果ページのHTMLを取得

### レースID生成関数の定義
西暦年を引数にその年のレースIDを生成する

In [2]:
def makeRaceID(tgtyear):
    """
    読み込む対象のraceIDを生成する関数
    """
    raceIDList=[]
    for place in range(1,11,1):
        for kai in range(1,6,1):
            for day in range(1,10,1):
                for race in range(1,13,1):
                    raceID = str(tgtyear).zfill(4) + str(place).zfill(2) + str(kai).zfill(2) + str(day).zfill(2) + str(race).zfill(2)
                    raceIDList.append(raceID)
    return raceIDList

### HTML取得用関数の定義
レース結果ページのHTMLをbinファイルに保存する

In [3]:
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Edg/115.0.0.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
]

def screpingRaceHTML(raceIDList,skip=True):
    """
    receIDリストに含まれるレースの結果ページのHTMLをスクレイピングする
    """
    cnt = 1
    for raceID in raceIDList:
        print("\r" + "現在 [raceID:"+str(raceID) + "] のスクレイピングを実行中(" + str(cnt) + "/" + str(len(raceIDList)) + ")",end="")
        if skip and os.path.isfile(f"../data/html/race/{raceID}.txt"):
            cnt += 1
            continue
        else:
            url = "https://db.netkeiba.com/race/" + raceID
            headers = {'User-Agent': random.choice(USER_AGENTS)}
            html = requests.get(url, headers=headers)
            html.encoding = "EUC-JP"
            time.sleep(3)

            try:
                test = pd.read_html(StringIO(html.text))[0]
                with open(f"../data/html/race/{raceID}.txt","w") as f:
                    f.write(html.text)
                cnt += 1

            except IndexError:
                cnt += 1
                continue
            
    return

### 実行

In [ ]:
start=2018
end=2019

for year in range(start,end + 1):
    raceIDList = makeRaceID(year)
    screpingRaceHTML(raceIDList)

    print(f"{year}年のHTMLの取得完了")

現在 [raceID:201910050912] のスクレイピングを実行中(5400/5400)

## RAWデータの作成

### RAWデータ作成用関数の定義

In [6]:
def makeRawDataRace(files,skip:bool = True):
    """
    html/raceに格納されているhtmlからレース結果/レース情報/払い戻し情報のrawデータを作成する関数
    """

    cnt = 1
    for file in files:
        raceID = re.sub(r"\D+", "", str(file.split("\\")[-1:]))
        if skip == False or os.path.isfile(f"../data/rawData/raceResults/{raceID}.pickle") == False or os.path.isfile(f"../data/rawData/raceInfos/{raceID}.pickle") == False or os.path.isfile(f"../data/rawData/return/{raceID}.pickle") == False:
            
            #共通処理======================================================================================================================
            with open(file,"r") as f:
                html = f.read()

            print("\r" + "現在 [raceID:"+str(raceID) + "] のrawデータを作成中(" + str(cnt) + "/" + str(len(files)) + ")",end="")

            soup = BeautifulSoup(html) #htmlをsoupオブジェクトへ
            
            #receResultsに関する処理=======================================================================================================
            if skip == False or os.path.isfile(f"../data/rawData/raceResults/{raceID}.pickle") == False:
                raceResults = pd.DataFrame() #最終出力データセットの定義
                dfRace = pd.read_html(StringIO(html))[0] #HTMLからレース結果を取得し、DataFrameへ

                dfRace["raceID"] = raceID

                horseIDList = []
                horseAList = soup.find("table",attrs={"summary":"レース結果"}).find_all("a",attrs={"href":re.compile("^/horse")})
                for a in horseAList:
                    horseID = re.findall("\d+",a["href"])
                    horseIDList.append(horseID[0])
                    
                dfRace["horseID"] = horseIDList

                jockeyIDList = []
                jockeyAList = soup.find("table",attrs={"summary":"レース結果"}).find_all("a",attrs={"href":re.compile("^/jockey")})
                for a in jockeyAList:
                    jockeyID = re.findall("\d+",a["href"])
                    jockeyIDList.append(jockeyID[0])

                dfRace["jockeyID"] = jockeyIDList

                raceResults = pd.concat([raceResults,dfRace])

                raceResults.to_pickle(f"../data/rawData/raceResults/{raceID}.pickle")

            #receInfosに関する処理==========================================================================================================
            if skip == False or os.path.isfile(f"../data/rawData/raceInfos/{raceID}.pickle") == False:
                raceInfos = pd.DataFrame() #最終出力データセットの定義
                text = soup.find("div",attrs={"class" : "data_intro"}).find_all("p")[0].text + soup.find("div",attrs={"class" : "data_intro"}).find_all("p")[1].text
                info = re.findall("\w+",text)

                infoDict = {}
                infoDict["raceID"] = raceID
                infoDict["raceName"] = soup.find("div",attrs={"class" : "data_intro"}).find_all("h1")[0].text
                for text in info:
                    if text in ["芝","ダート"]:
                        infoDict["raceType"] = text
                    if "障" in text:
                        infoDict["raceType"] = "障害"
                    if "m" in text:
                        infoDict["course_len"] = re.findall("\d+",text)[0]
                    if text in ["良","稍重","重","不良"]:
                        infoDict["groundState"] = text
                    if text in ["曇","晴","雨","小雨","小雪","雪"]:
                        infoDict["weather"] = text
                    if "年" in text :
                        infoDict["date"] = text

                raceInfos = pd.DataFrame(infoDict,index=[0])
                #raceInfos = pd.concat([raceInfos, dfInfo], ignore_index=True)

                raceInfos.to_pickle(f"../data/rawData/raceInfos/{raceID}.pickle")
            
            #returnTableに関する処理========================================================================================================
            #同着の時にイレギュラーパターンがあるからそれを処理する機能の追加必須
            if skip == False or os.path.isfile(f"../data/rawData/return/{raceID}.pickle") == False:
                html2 = html.replace("<br />","or")
                tables = pd.read_html(StringIO(html2))
                returnTable = pd.concat([tables[1],tables[2]])
                returnTable.reset_index(drop=True,inplace=True)
                
                betList = returnTable[0].unique()
                for betting in betList:
                #for rowNo in range(len(returnTable)):
                    tgt = returnTable[returnTable[0] == betting]
                    if "or" in tgt[1].to_string(index=False):        
                        for i in [1,2,3]:
                            tgtstring = tgt[i].to_string(index=False)
                            tgtlist = tgtstring.split("or")
                            tgtdf = pd.DataFrame(tgtlist)
                            tgtdf.rename(columns={0:i},inplace=True)
                            if i == 1:
                                returnFin = tgtdf
                            else:
                                returnFin = pd.merge(returnFin,tgtdf,left_index=True, right_index=True,how="left")

                            returnFin[0] = betting

                            returnTable = returnTable[returnTable[0] != betting]
                            returnTable = pd.concat([returnTable,returnFin])

                returnTable.reset_index(drop=True,inplace=True)
                returnTable.rename(columns={0:"betting",1:"results",2:"payout",3:"popularity"},inplace=True)

                returnTable["payout"] = returnTable["payout"].str.replace(",","").astype(int)
                returnTable["payoutRate"] = returnTable["payout"] / 100
                returnTable["raceID"] = raceID


                returnTable.to_pickle(f"../data/rawData/return/{raceID}.pickle")

        cnt+=1      
    return

### 実行

In [7]:
start=2018
end=2024
for year in range(start,end + 1):
    files = glob.glob(f"..\\data\\html\\race\\{year}*")
    makeRawDataRace(files,skip=True)
    
    print("\r" + f"{year}年のデータ作成完了")

2018年のデータ作成完了1810020912] のrawデータを作成中(3310/3310)
2019年のデータ作成完了1910020912] のrawデータを作成中(3308/3308)
2020年のデータ作成完了2010020812] のrawデータを作成中(3252/3252)
2021年のデータ作成完了2110040812] のrawデータを作成中(3096/3096)
2022年のデータ作成完了2210040812] のrawデータを作成中(3096/3096)
2023年のデータ作成完了2310030812] のrawデータを作成中(3312/3312)
2024年のデータ作成完了2410030812] のrawデータを作成中(3131/3131)


# 馬情報(過去成績/血統)のRAWデータ作成

## 馬情報ページのHTML取得

### horseIDリスト作成用関数の定義

In [2]:
def makehorseIDList(files):
    """
    filesに指定されたレース結果からユニークなhorseIDのリストを作成する
    """
    horseIDList = []

    cnt = 1
    for file in files:
        print("\r" + f"horseIDListの作成中({cnt}/{len(files)})",end="")
        filename = str(file.split("/")[-1:])
        raceID = re.sub(r"\D+", "", filename)

        horseIDTmp = pd.read_pickle(f"../data/rawData/raceResults/{raceID}.pickle")
        horseIDTmp2 = horseIDTmp["horseID"].unique()
        horseIDList.extend(horseIDTmp2)
        cnt+=1
    
    horseIDListDF = pd.DataFrame(horseIDList)
    horseIDListFin = horseIDListDF[0].unique()

    print("\n" + f"{len(horseIDListFin)}頭分のhorseIDListを作成しました")
    return horseIDListFin

### HTML取得用関数の定義

In [ ]:
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Edg/115.0.0.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
]

def screpingHorsePEDHTML(horseIDList,skipResult=True,skipPED=True):
    """
    horseIDリストに含まれる結果/血統ページのHTMLをスクレイピングする
    """
    cnt = 1
    for horseID in horseIDList:
        #馬の過去成績のHTML取得=====================================================
        if skipResult == False or os.path.isfile(f"../data/html/horse/{horseID}.txt") == False :
            print("\r" + "現在 [horseID:"+str(horseID) + "] の過去戦績のHTML取得を実行中(" + str(cnt) + "/" + str(len(horseIDList)) + ")",end="")
            url = "https://db.netkeiba.com/horse/" + horseID
            headers = {'User-Agent': random.choice(USER_AGENTS)}
            html = requests.get(url, headers=headers)
            html.encoding = "EUC-JP"

            with open(f"../data/html/horse/{horseID}.txt","w") as f:
                f.write(html.text)
            
            time.sleep(2)

        #馬の血統表情報のHTML取得===================================================
        if skipPED == False or os.path.isfile(f"../data/html/ped/{horseID}.txt") == False :
            print("\r" + "現在 [horseID:"+str(horseID) + "] の血統情報のHTML取得を実行中(" + str(cnt) + "/" + str(len(horseIDList)) + ")",end="")
            url2 = "https://db.netkeiba.com/horse/ped/" + horseID
            headers2 = {'User-Agent': random.choice(USER_AGENTS)}
            html2 = requests.get(url2, headers=headers2)
            html2.encoding = "EUC-JP"

            with open(f"../data/html/ped/{horseID}.txt","w") as f:
                f.write(html2.text)
            
            time.sleep(2)

        cnt += 1  
    return
#過去成績と血統は別々でスキップするようにしたほうが良さそう

### HTML取得 & rawデータ作成クラスの定義
`makeRawData.py`の`makeHorseData`クラスとしてモジュール化済み

In [ ]:
import my_snipets
import random
import os
import requests
import time
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup
import re

class makeHorseData:
    """
    馬データを作成するクラス  
    `horseIDList`:取得対象となるhorseIDのリスト

    --- 
    `getHorseHTML(関数)`:`horseIDList`の中にある馬の過去成績と血統表のHTMLを取得し、txtファイルとして保存する
    ---
    - `skipResult:bool` => 過去成績のHTML取得をスキップするかどうか
    - `skipPED:bool` => 血統表のHTML取得をスキップするかどうか  
    ---
    `makeHorseRawData(関数)`:`horseIDList`の中にある馬の過去成績と血統表のRawデータを作成し、pickleファイルとして保存する  
    ---
    - `skipResult:bool` => 過去成績のRawデータ作成をスキップするかどうか
    - `skipPED:bool` =>血統表のRawデータ作成をスキップするかどうか  
    """
    def __init__(self, horseIDList):
        self.horseIDList = horseIDList
    
    def getHorseHTML(self, skipResult=False,skipPED=True):
        cnt = 1
        for horseID in self.horseIDList:
            #馬の過去成績のHTML取得=====================================================
            if skipResult == False or os.path.isfile(f"../data/html/horse/{horseID}.txt") == False :
                print("\r" + "現在 [horseID:"+str(horseID) + "] の過去戦績のHTML取得を実行中(" + str(cnt) + "/" + str(len(self.horseIDList)) + ")",end="")
                url = "https://db.netkeiba.com/horse/" + horseID
                headers = {'User-Agent': random.choice(my_snipets.makeUserAgents())}
                html = requests.get(url, headers=headers)
                html.encoding = "EUC-JP"

                with open(f"../data/html/horse/{horseID}.txt","w") as f:
                    f.write(html.text)
                
                time.sleep(2)

            #馬の血統表情報のHTML取得===================================================
            if skipPED == False or os.path.isfile(f"../data/html/ped/{horseID}.txt") == False :
                print("\r" + "現在 [horseID:"+str(horseID) + "] の血統情報のHTML取得を実行中(" + str(cnt) + "/" + str(len(self.horseIDList)) + ")",end="")
                url2 = "https://db.netkeiba.com/horse/ped/" + horseID
                headers2 = {'User-Agent': random.choice(my_snipets.makeUserAgents())}
                html2 = requests.get(url2, headers=headers2)
                html2.encoding = "EUC-JP"

                with open(f"../data/html/ped/{horseID}.txt","w") as f:
                    f.write(html2.text)
                
                time.sleep(2)

            cnt += 1

        return

    def makeHorseRawData(self,skipResult=False,skipPED=True):
        """
        ファイルリスト内のHTMLから馬の過去成績のRawデータを作成する関数
        """
        columns = ["F","FF","FFF","FFFF","FFFFF","FFFFM","FFFM","FFFMF","FFFMM","FFM","FFMF","FFMFF","FFMFM","FFMM","FFMMF","FFMMM","FM","FMF","FMFF","FMFFF","FMFFM","FMFM","FMFMF","FMFMM","FMM","FMMF","FMMFF","FMMFM","FMMM","FMMMF","FMMMM",
                   "M","MF","MFF","MFFF","MFFFF","MFFFM","MFFM","MFFMF","MFFMM","MFM","MFMF","MFMFF","MFMFM","MFMM","MFMMF","MFMMM","MM","MMF","MMFF","MMFFF","MMFFM","MMFM","MMFMF","MMFMM","MMM","MMMF","MMMFF","MMMFM","MMMM","MMMMF","MMMMM"]

        cnt = 1
        for horseID in self.horseIDList:
            #馬の過去成績のrawデータを作成================================================            
            if skipResult == False or os.path.isfile(f"../data/rawData/horse/{horseID}.pickle") == False:
                print("\r" + "現在 [horseID:"+str(horseID) + "] の過去戦績のrawデータを作成中(" + str(cnt) + "/" + str(len(self.horseIDList)) + ")",end="")

                # htmlが格納されたファイルを開く
                with open(f"../data/html/horse/{horseID}.txt","r") as f:
                    html = f.read()

                #htmlから過去戦績のデータを抽出
                contents = pd.read_html(StringIO(html))
                if contents[2].columns[0] == "日付":
                    horseResults = contents[2]
                else:
                    horseResults = contents[3]
                
                #データフレームをpickle形式で保存
                horseResults.to_pickle(f"../data/rawData/horse/{horseID}.pickle")

            if skipPED == False or os.path.isfile(f"../data/rawData/ped/{horseID}.pickle") == False:
                print("\r" + "現在 [horseID:"+str(horseID) + "] の血統情報データを作成中(" + str(cnt) + "/" + str(len(self.horseIDList)) + ")",end="")

                with open(f"../data/html/ped/{horseID}.txt","r") as f: #ファイルを開く
                    html = f.read()       

                soup = BeautifulSoup(html,"html.parser") #htmlをsoupオブジェクトへ

                contents = soup.find("table",attrs={"summary":"5代血統表"}).find_all("a",attrs={"href":re.compile(r"^/horse/\d+")}) #血統表の馬名の部分を抽出

                pedHorseNameList = []
                for text in contents:
                    text2 = str(text.text)
                    text3 = text2.split("\n")[0]
                    pedHorseNameList.append(text3)

                PEDFinDF = pd.DataFrame(pedHorseNameList).T
                for i in range(len(columns)):
                    PEDFinDF.rename(columns={int(f"{i}"):f"{columns[i]}"},inplace=True)
                PEDFinDF.rename(index={0:str(horseID)},inplace=True)

                PEDFinDF.to_pickle(f"../data/rawData/ped/{horseID}.pickle")

            cnt += 1
        return

### 実行

In [32]:
files = glob.glob("../data/rawData/raceResults/*")
print(f"{len(files)}レース分の結果ファイルのリストを作成しました")

22505レース分の結果ファイルのリストを作成しました


In [33]:
horseIDList = makehorseIDList(files)

horseIDListの作成中(22505/22505)
39839頭分のhorseIDListを作成しました


In [57]:
screpingHorsePEDHTML(horseIDList,skipResult=True,skipPED=True)

現在 [horseID:2018103550] の血統情報のHTML取得を実行中(21846/39839)

## RAWデータの作成

### 過去成績のRAWデータ作成用関数の定義

In [8]:
def makeRawDataHorse(files,skip=False):
    """
    ファイルリスト内のHTMLから馬の過去成績のRawデータを作成する関数
    """
    cnt = 1
    for file in files:
        horseID = re.sub(r"\D+","", str(file.split("/")[-1:]))
        
        if skip == False or os.path.isfile(f"../data/rawData/horse/{horseID}.pickle") == False:
            with open(file,"r") as f:
                html = f.read()

            print("\r" + "現在 [horseID:"+str(horseID) + "] のデータを作成中(" + str(cnt) + "/" + str(len(files)) + ")",end="")
            contents = pd.read_html(StringIO(html))
            if contents[2].columns[0] == "日付":
                horseResults = contents[2]
            else:
                horseResults = contents[3]
            
            horseResults.to_pickle(f"../data/rawData/horse/{horseID}.pickle")

        cnt += 1
    return

### 実行

In [9]:
files = glob.glob("../data/html/horse/*")
print(f"対象頭数:{len(files)}")

makeRawDataHorse(files,skip=True)

対象頭数:39839
現在 [horseID:2022110153] のデータを作成中(39839/39839)

### 血統情報のRAWデータ作成用関数の定義

In [ ]:
def makeRawDataPED(files,skip=True):
    """
    ファイルリスト内のHTMLから血統情報のRawデータを作成する関数
    """

    columns = ["F","FF","FFF","FFFF","FFFFF","FFFFM","FFFM","FFFMF","FFFMM","FFM","FFMF","FFMFF","FFMFM","FFMM","FFMMF","FFMMM","FM","FMF","FMFF","FMFFF","FMFFM","FMFM","FMFMF","FMFMM","FMM","FMMF","FMMFF","FMMFM","FMMM","FMMMF","FMMMM",
               "M","MF","MFF","MFFF","MFFFF","MFFFM","MFFM","MFFMF","MFFMM","MFM","MFMF","MFMFF","MFMFM","MFMM","MFMMF","MFMMM","MM","MMF","MMFF","MMFFF","MMFFM","MMFM","MMFMF","MMFMM","MMM","MMMF","MMMFF","MMMFM","MMMM","MMMMF","MMMMM"]
    cnt = 1
    for file in files:
        horseID = re.sub(r"\D+","", str(file.split("/")[-1:]))

        if skip == False or os.path.isfile(f"../data/rawData/ped/{horseID}.pickle") == False:
            with open(file,"r") as f: #ファイルを開く
                html = f.read()       

            print("\r" + "現在 [horseID:"+str(horseID) + "] のデータを作成中(" + str(cnt) + "/" + str(len(files)) + ")",end="")
            soup = BeautifulSoup(html,"html.parser") #htmlをsoupオブジェクトへ

            contents = soup.find("table",attrs={"summary":"5代血統表"}).find_all("a",attrs={"href":re.compile(r"^/horse/\d+")}) #血統表の馬名の部分を抽出

            pedHorseNameList = []
            for text in contents:
                text2 = str(text.text)
                text3 = text2.split("\n")[0]
                pedHorseNameList.append(text3)

            PEDFinDF = pd.DataFrame(pedHorseNameList).T
            for i in range(len(columns)):
                PEDFinDF.rename(columns={int(f"{i}"):f"{columns[i]}"},inplace=True)
            PEDFinDF.rename(index={0:str(horseID)},inplace=True)

            PEDFinDF.to_pickle(f"../data/rawData/ped/{horseID}.pickle")

        cnt += 1
    return 

### 実行

In [11]:
files = glob.glob("../data/html/ped/*")
print(f"対象頭数:{len(files)}")

makeRawDataPED(files)

対象頭数:39839
現在 [horseID:2022110153] のデータを作成中(39839/39839)

騎手の過去成績どうする?